# ⚡ Khởi Chạy Kaggle GPU Worker (Dual Tesla T4) cho Multi-Agent RAG
**Tự động tải mô hình vào 2x GPU T4 & Xuất duy nhất 1 link Cloudflare Tunnel để kết nối với Web UI Laptop (`http://localhost:8000/demo2/`)**


In [ ]:
# BƯỚC 1: TỰ ĐỘNG CLONE DỰ ÁN TỪ GITHUB & CÀI ĐẶT THƯ VIỆN
import os
import subprocess

REPO_URL = "https://github.com/ducmanh-jr/Multi-Agent_RAG.git"
PROJECT_DIR = "/content/Multi-Agent_RAG" if os.path.exists("/content") else "/kaggle/working/Multi-Agent_RAG"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=False)

os.chdir(PROJECT_DIR)
!pip install -q -r requirements.txt
!pip install -q fastapi uvicorn pyngrok gdown transformers peft bitsandbytes accelerate sentence-transformers rank_bm25 pymupdf torch arxiv
print("[OK] Đã cài đặt xong môi trường tại:", os.getcwd())


In [ ]:
# BƯỚC 2: TỰ ĐỘNG NẠP TRỌNG SỐ LORA TỪ GOOGLE DRIVE HOẶC TRONG REPO
import gdown

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1upmXzTIlxpcCVGU8Vp9ZGV3xNa1yu-Q6?usp=sharing"
DRIVE_FOLDER_ID = "1upmXzTIlxpcCVGU8Vp9ZGV3xNa1yu-Q6"
DRIVE_FILE_ID = ""  # (Tùy chọn) Điền File ID của weights_bundle.zip trên Google Drive nếu muốn tải bản mới nhất
WEIGHTS_DIR = os.path.join(PROJECT_DIR, "training_lab", "weights_drive_bundle")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

if DRIVE_FILE_ID:
    zip_out = os.path.join(WEIGHTS_DIR, "weights_bundle.zip")
    gdown.download(id=DRIVE_FILE_ID, output=zip_out, quiet=False)
    !unzip -q -o {zip_out} -d {WEIGHTS_DIR}
    print("[OK] Đã tải và giải nén weights_bundle.zip từ Google Drive!")
elif os.path.exists(os.path.join(WEIGHTS_DIR, "weights_bundle.zip")):
    !unzip -q -o {WEIGHTS_DIR}/weights_bundle.zip -d {WEIGHTS_DIR}
    print("[OK] Đã giải nén weights_bundle.zip có sẵn!")
else:
    print("[OK] Sử dụng trọng số có sẵn trong training_lab/weights_drive_bundle/")


In [ ]:
# BƯỚC 3: KHỞI ĐỘNG GPU WORKER (DUAL T4) & PHÁT LINK CLOUDFLARE TUNNEL
import time
import re
import subprocess
import urllib.request
import json

# 1. Cài đặt cloudflared
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# 2. Khởi chạy GPU Worker Server ở cổng 8000
os.environ["WEIGHTS_DIR"] = WEIGHTS_DIR
print("🚀 [1/3] Đang khởi động GPU Worker & nạp mô hình vào Dual Tesla T4...")
print("⏳ (Quá trình tải và nạp mô hình Qwen vào VRAM mất khoảng 1-2 phút lần đầu, vui lòng đợi...)")

worker_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "kaggle_server:app", "--app-dir", "kaggle", "--host", "0.0.0.0", "--port", "8000"],
    cwd=PROJECT_DIR,
)

# 3. Đợi máy chủ GPU nạp xong mô hình hoàn toàn (Healthcheck tự động)
is_ready = False
for attempt in range(120):
    try:
        resp = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        if resp.status == 200:
            data = json.loads(resp.read().decode())
            print(f"✅ [2/3] Mô hình đã nạp xong vào VRAM! GPU Count: {data.get('gpu_count', 2)}")
            is_ready = True
            break
    except Exception:
        time.sleep(2)

if not is_ready:
    print("⚠️ Đang tiếp tục mở Cloudflare tunnel...")

# 4. Mở Cloudflare Tunnel ra Internet cho GPU Worker
print("🌐 [3/3] Đang mở kết nối Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

print("\n" + "=" * 78)
print("🎉 KAGGLE DUAL TESLA T4 GPU WORKER ĐÃ SẴN SÀNG KẾT NỐI!")
print("=" * 78)
print(f"👉 ĐƯỜNG LINK DUY NHẤT:  {public_url}")
print(f"👉 DÁN VÀO Ô:            GPU Kaggle Tunnel (Ngrok): trên Web UI (localhost:8000)")
print(f"👉 RỒI BẤM NÚT:          'Lưu' để kích hoạt 2x GPU Tesla T4")
print("=" * 78 + "\n")

try:
    worker_proc.wait()
except KeyboardInterrupt:
    worker_proc.terminate()
    tunnel_proc.terminate()
